# Ramen：联合语义自适应 3DGS 对比测试

运行时请选择 **L4 GPU**。`pilot` 用于验证全链路；`full` 用于正式对比。测试集采用 LERF-Mask Ramen 官方 `test_*.jpg` 和 `test_mask`，输出 PSNR、SSIM、mIoU、Boundary-IoU、高斯总数和三级高斯数量。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 运行时 > 更改运行时类型 中选择 L4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%cd /content
![ -d /content/gaussian-splatting/.git ] && git -C /content/gaussian-splatting pull --ff-only || git clone --recursive https://github.com/Xuyw041006-arch/gaussian-splatting.git /content/gaussian-splatting
%cd /content/gaussian-splatting
!git submodule update --init --recursive
!pip -q install plyfile open-clip-torch scikit-learn ftfy regex opencv-python-headless
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install ./submodules/diff-gaussian-rasterization ./submodules/simple-knn ./submodules/fused-ssim
!python -m unittest discover -s tests -p 'test_*.py' -v
!python -m compileall -q scene semantic scripts train.py preprocess_semantics.py

In [ ]:
%cd /content
!test -f ramen.zip || wget -q --show-progress https://huggingface.co/mqye/Gaussian-Grouping/resolve/main/data/lerf_mask/ramen.zip
!mkdir -p /content/ramen_benchmark
!test -d /content/ramen_benchmark/ramen || unzip -q ramen.zip -d /content/ramen_benchmark
!test -f sam_vit_h_4b8939.pth || wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
print('数据集和 SAM ViT-H 检查点已就绪')

In [ ]:
MODE = 'pilot'  # 改成 'full' 后重新运行本单元
ITERATIONS = 1500 if MODE == 'pilot' else 30000
SEMANTIC_ITERATIONS = 300 if MODE == 'pilot' else 5000
SEMANTIC_START = 500 if MODE == 'pilot' else 1000
OUTPUT = f'/content/ramen_results_{MODE}'
from pathlib import Path
REUSE_PREPROCESS = (Path('/content/ramen_benchmark/ramen') / 'semantic_meta.npz').is_file()
SKIP_PREPROCESS = '--skip_preprocess' if REUSE_PREPROCESS else ''
print(MODE, ITERATIONS, SEMANTIC_ITERATIONS, SEMANTIC_START, OUTPUT, 'reuse:', REUSE_PREPROCESS)

In [ ]:
%cd /content/gaussian-splatting
!python scripts/run_ramen_benchmark.py --scene /content/ramen_benchmark/ramen --sam_checkpoint /content/sam_vit_h_4b8939.pth --output_root {OUTPUT} --iterations {ITERATIONS} --semantic_iterations {SEMANTIC_ITERATIONS} --semantic_start {SEMANTIC_START} {SKIP_PREPROCESS}
import json
from pathlib import Path
comparison = json.loads((Path(OUTPUT) / 'comparison.json').read_text())
comparison

In [ ]:
from IPython.display import display
from PIL import Image
predictions = sorted((Path(OUTPUT) / 'eval_joint').glob('*.png'))[:6]
for path in predictions:
    print(path.name)
    display(Image.open(path))